# Chapter 11: Fine-Tuning BERT for Text Classification

Fine-tuning means taking a pre-trained model and continuing to train it on your specific task with a small labelled dataset.

**Why fine-tune?**
- Pre-trained models already understand language
- You only need to teach the task, not language itself
- A few thousand examples is often enough

## Setup

In [ ]:
# !pip install transformers datasets torch scikit-learn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score
print('Ready')

## Step 1 — Load dataset (SST-2 sentiment)

In [ ]:
dataset = load_dataset("sst2")
print(dataset)
print("\nSample:")
for ex in dataset["train"].select(range(3)):
    print(f"  label={ex['label']} text={ex['sentence'][:80]}")

## Step 2 — Tokenise

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenisation done")
print("Sample keys:", list(tokenized["train"][0].keys()))

## Step 3 — Load model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 4 — Define training arguments

In [ ]:
args = TrainingArguments(
    output_dir="/tmp/distilbert-sst2",
    evaluation_strategy="epoch",
    save_strategy="no",
    num_train_epochs=1,          # 1 epoch for demo — use 3 for real training
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

## Step 5 — Train

In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"].select(range(4000)),   # subset for speed
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)
trainer.train()

## Step 6 — Evaluate and predict

In [ ]:
results = trainer.evaluate()
print("Validation accuracy:", results["eval_accuracy"])

# Inference on custom text
from transformers import pipeline
clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
test_sentences = [
    "This film is absolutely brilliant!",
    "Worst movie I have ever seen.",
    "It was okay, nothing special.",
]
for s in test_sentences:
    pred = clf(s)[0]
    label = "POSITIVE" if pred["label"] == "LABEL_1" else "NEGATIVE"
    print(f"  [{label} {pred['score']:.3f}] {s}")

## Summary

Fine-tuning recipe:
1. **Load** pretrained model + tokenizer
2. **Tokenise** your dataset
3. **Train** with `Trainer` (handles batching, optimisation, evaluation)
4. **Evaluate** on a held-out set

With 1 epoch on 4k examples, DistilBERT typically reaches ~90% accuracy on SST-2. With 3 epochs and the full dataset, ~92%.